In [1]:
from config_file import *
sys.path.append('./problems')
from train import *

## Some notes
I had to repeat experiments a few times, but at this point I ran 3 seeds depth 2 for 4 different models.

However I found that the features used to make T1 and T2 seemed switched, so I investigated that. With a lot of sanity checks (not limited to the ones) below the corresponding header at bottom.

Then however I also saw that the best fold was hard-coded into the multifeature.py document, so I had to change that and rerun the experiments (comp comp, part none, none none). In the document xai_v2_replace_fts.ipynb I'll go ahead and continue with the one that did run correctly so I can plug in new results quickly.

In [2]:
# Load the label data
labels = pd.read_csv(DATA_DIR + '/fusion_labels.csv')
tab = pd.read_csv(DATA_DIR + '/tab_data.csv')
tab.drop(columns=['A', 'B', 'C', 'id', 'Feature14', 'Feature13', 'Feature12', 'Feature11', 'Feature10'], inplace=True)

# Combine the tabular data and labels into a single DataFrame for training
data = pd.concat([tab, labels], axis=1)
data['y'] = get_y(data, 'fusion')

seed = set_seed(SEED) # Set the random seed for reproducibility
nas = {"img_fts": IMG_FTS, "tab_fts": TAB_FTS}
print(nas)

{'img_fts': 2, 'tab_fts': 2}


In [3]:
model = multifix_net_test(nas, OUT_SIZE)
loss_fn = nn.BCELoss()

In [4]:
# Split data into training and testing folds
for i, (train_idx, test_idx) in enumerate(split_data(data, seed)):
    if i == BEST_FOLD:
        train_loader, val_loader, test_loader = get_dataloaders(data, train_idx, test_idx, seed)
        #model, loss_fn = initialise_model(i, nas, True) # initialise the model according to prediction type and training procedure
        wts_dir = save_dir = '/export/scratch2/ima/MultiFIX_GECCO25_code' + MODEL_DIR[1:] + INPUT + '_' + TRAINING + '_' + WTS + '_Freeze_' + str(TEMP_FREEZE) + '.pth'
        #wts_dir = save_dir = REPO_DIR + MODEL_DIR[1:] + INPUT + '_' + TRAINING + '_' + WTS + '_Freeze_' + str(TEMP_FREEZE) + '.pth'
        model.load_state_dict(torch.load(wts_dir), strict=False)
        loss, auroc, bacc = eval_model(model, test_loader, loss_fn)  # Evaluate the model
        break

In [5]:
# Check Evaluation Metrics
print("*** BEST FOLD METRICS ***")
print("* Loss = ", loss)
print("* AUROC = ", auroc)
print("* BAcc = ", bacc)

*** BEST FOLD METRICS ***
* Loss =  0.0554846932401415
* AUROC =  0.9979707792207793
* BAcc =  0.9898538961038961


# Generate files for GP-GOMEA
* we need the tabular features and the engineered tabular features to feed to GPG
* we also safe files with the engineered features (img and tab) and the true labels

doing this for :
comp comp (done)
none none (done)
part part (done)
part none (done)

In [6]:
# check before making gp file:
print(f'LR = {LR}, WD = {WD}, BEST_FOLD = {BEST_FOLD}')
print(f'GP DIR = {GP_DIR}')
print_problem()
print(f'TRAINING = {TRAINING}')

LR = 0.001, WD = 0.0, BEST_FOLD = 2
GP DIR = ./gp_files/Multifeature/ft_comp_comp_single_False/
Multifeature Problem: OR(AND(circle,A), AND(!triangle, B)); TRAINING =  ft_comp_comp ; INPUT =  fusion ; OUT_SIZE =  1 ; WTS =  single ; TEMP_FREEZE =  False
TRAINING = ft_comp_comp


In [1]:
import pandas as pd
import numpy as np

save_csv = False

if save_csv:

    if not os.path.exists(GP_DIR):
        os.makedirs(GP_DIR)

    dataloaders = [('train', train_loader), ('test', test_loader)]

    with torch.no_grad():
        for split, loader in dataloaders:
            # Initialize tab_fts, I, T, Y arrays
            tab_fts, all_I, all_T, all_Y = [], [], [], []
            
            # Iterate through the dataloader and extract the features
            for i, (img, tab, _, _, y) in enumerate(loader):
                img, tab, y = img.to(DEVICE, dtype=torch.float), tab.to(DEVICE, dtype=torch.float), y.to(DEVICE, dtype=torch.float)
                
                # Get the outputs from model blocks
                I_c = model.img_c_block(img, tab)
                I_t = model.img_t_block(img, tab)
                I = torch.concat((I_c, I_t),axis = 1)

                T_a = model.tab_a_block(img, tab)
                T_b = model.tab_b_block(img, tab)
                T = torch.concat((T_a, T_b),axis = 1)

                # if multiclass
                # Y = np.argmax(y.cpu().numpy(), axis=1)
                
                
                # Convert tensors to numpy and handle dimensions carefully
                tab_fts.append(tab.cpu().numpy())           # Keep tab_fts as is
                all_I.append(I.cpu().numpy())               # Append I features (variable column sizes)
                all_T.append(T.cpu().numpy())               # Append T features
                all_Y.append(y.cpu().numpy())               # Append labels

            # Convert lists to NumPy arrays and ensure consistent shape by concatenating along axis 0
            tab_fts_np = np.concatenate(tab_fts, axis=0)     # Stack rows for tab_fts
            all_I_np = np.concatenate(all_I, axis=0)         # Stack rows for all_I
            all_T_np = np.concatenate(all_T, axis=0)         # Stack rows for all_T
            all_Y_np = np.concatenate(all_Y, axis=0)         # Stack rows for all_Y

            # Ensure all_Y has a consistent shape, even if it has only one column
            if all_Y_np.ndim == 1:
                all_Y_np = all_Y_np[:, np.newaxis]           # Make sure all_Y is 2D (n, 1)

            # Create DataFrame for the first file (all_I, all_T, all_Y)
            df_1 = pd.DataFrame(
                data=np.hstack([all_I_np, all_T_np, all_Y_np]),  # Horizontally concatenate arrays
                columns=[f'I{i+1}' for i in range(all_I_np.shape[1])] +  # Name all_I columns
                        [f'T{i+1}' for i in range(all_T_np.shape[1])] +  # Name all_T columns
                        ['Y']  # Name the target column (all_Y)
            )
            
            # Create DataFrame for the second file (tab_fts, all_T)
            df_2 = pd.DataFrame(
                data=np.hstack([tab_fts_np, all_T_np]),  # Horizontally concatenate tab_fts and all_T
                columns=[f'tab_fts_col_{i+1}' for i in range(tab_fts_np.shape[1])] +  # Name tab_fts columns
                        [f'T{i+1}' for i in range(all_T_np.shape[1])]  # Name all_T columns
            )

            # Save both DataFrames as CSV files
            file_1 = GP_DIR + f'{split}_dl_I_T_Y.csv'
            df_1.to_csv(file_1, index=False)
            print(f"Saved {file_1}!")
            
            file_2 = GP_DIR + f'{split}_tab_T.csv'
            df_2.to_csv(file_2, index=False)
            print(f"Saved {file_2}!")

    print(nas)

# Make GT GP files

In [32]:
b

tensor([[1., 1., 1.]], device='cuda:0')

In [38]:
import pandas as pd
import numpy as np

save_csv = True

if save_csv:

    if not os.path.exists(GP_DIR):
        os.makedirs(GP_DIR)

    dataloaders = [('train', train_loader), ('test', test_loader)]

    with torch.no_grad():
        for split, loader in dataloaders:
            # Initialize tab_fts, I, T, Y arrays
            tab_fts, all_a, all_b, all_Y = [], [], [], []
            
            # Iterate through the dataloader and extract the features
            for i, (_, tab, a, b, y) in enumerate(loader):
                a, b, y = a.to(DEVICE, dtype=torch.float), b.to(DEVICE, dtype=torch.float), y.to(DEVICE, dtype=torch.float)          
                
                # Convert tensors to numpy and handle dimensions carefully
                tab_fts.append(tab.cpu().numpy())           # Keep tab_fts as is
                all_a.append(a[:, [0, 2]].cpu().numpy())               # Append I features (variable column sizes)
                all_b.append(b[:, [0, 1]].cpu().numpy())               # Append T features
                all_Y.append(y.cpu().numpy())               # Append labels

            # Convert lists to NumPy arrays and ensure consistent shape by concatenating along axis 0
            tab_fts_np = np.concatenate(tab_fts, axis=0)     # Stack rows for tab_fts
            all_a_np = np.concatenate(all_a, axis=0)         # Stack rows for all_I
            all_b_np = np.concatenate(all_b, axis=0)         # Stack rows for all_T
            all_Y_np = np.concatenate(all_Y, axis=0)         # Stack rows for all_Y

            # Ensure all_Y has a consistent shape, even if it has only one column
            if all_Y_np.ndim == 1:
                all_Y_np = all_Y_np[:, np.newaxis]           # Make sure all_Y is 2D (n, 1)

            # Create DataFrame for the first file (all_I, all_T, all_Y)
            df_1 = pd.DataFrame(
                data=np.hstack([all_a_np, all_b_np, all_Y_np]),  # Horizontally concatenate arrays
                columns=[f'I{i+1}' for i in range(all_a_np.shape[1])] +  # Name all_I columns
                        [f'T{i+1}' for i in range(all_b_np.shape[1])] +  # Name all_T columns
                        ['Y']  # Name the target column (all_Y)
            )

            file_1 = f'/export/scratch2/ima/MultiFIX_GECCO25_code/scripts/gp_files/Multifeature/gt_method1/{split}_I_T_Y.csv'
            df_1.to_csv(file_1, index=False)
            print(f"Saved {file_1}!")
            

Saved /export/scratch2/ima/MultiFIX_GECCO25_code/scripts/gp_files/Multifeature/gt_method1/train_I_T_Y.csv!
Saved /export/scratch2/ima/MultiFIX_GECCO25_code/scripts/gp_files/Multifeature/gt_method1/test_I_T_Y.csv!


In [39]:
# making new ground truth csv files

# method 1 (see above cell, same as original gp file creation)

# method 2
train_df = data.iloc[train_idx][['circle', 'triangle', 'A', 'B', 'y']].rename(columns = {'circle': 'I1', 'triangle': 'I2', 'A': 'T1', 'B': 'T2', 'y': 'Y'}).astype('float32')
test_df = data.iloc[test_idx][['circle', 'triangle', 'A', 'B', 'y']].rename(columns = {'circle': 'I1', 'triangle': 'I2', 'A': 'T1', 'B': 'T2', 'y': 'Y'}).astype('float32')

m2_train_file = f'/export/scratch2/ima/MultiFIX_GECCO25_code/scripts/gp_files/Multifeature/gt_method2/train_I_T_Y.csv'
m2_test_file = f'/export/scratch2/ima/MultiFIX_GECCO25_code/scripts/gp_files/Multifeature/gt_method2/test_I_T_Y.csv'

save = True

if save:
    train_df.to_csv(m2_train_file, index=False)
    test_df.to_csv(m2_test_file, index=False)

In [ ]:
# for instance for comp_comp best fold split
test_list = list(test_loader)
y = []
test_rows = {}

for i, feats in enumerate(test_list):
    i_feats = feats[-3].flatten().tolist()[:2]
    t_feats = feats[-2].flatten().tolist()[:2]
    y = feats[-1].flatten().tolist()
    test_rows[i] = i_feats + t_feats + y
    

# same for training
train_list = list(train_loader)
y = []
train_rows = {}

for i, feats in enumerate(train_list):
    
    
    i_feats = feats[-3].flatten().tolist()[:2]
    t_feats = feats[-2].flatten().tolist()[:2]
    y = feats[-1].flatten().tolist()
    train_rows[i] = i_feats + t_feats + y


gt_test_df = pd.DataFrame.from_dict(test_rows, orient = 'index', columns = ['I1', 'I2', 'T1', 'T2', 'Y'])
gt_train_df = pd.DataFrame.from_dict(train_rows, orient = 'index', columns = ['I1', 'I2', 'T1', 'T2', 'Y'])



ValueError: 5 columns passed, passed data had 36 columns

# Sanity checks (ignore)

In [ ]:
# sanity check: do newly generated files, match existing files
import pandas as pd
import numpy as np

dataloaders = [('train', train_loader)]#, ('test', test_loader)]

with torch.no_grad():
    for split, loader in dataloaders:
        # Initialize tab_fts, I, T, Y arrays
        tab_fts, all_I, all_T, all_Y = [], [], [], []
        
        # Iterate through the dataloader and extract the features
        for i, (img, tab, _, _, y) in enumerate(loader):
            img, tab, y = img.to(DEVICE, dtype=torch.float), tab.to(DEVICE, dtype=torch.float), y.to(DEVICE, dtype=torch.float)
            
            # Get the outputs from model blocks
            I_c = model.img_c_block(img, tab)
            I_t = model.img_t_block(img, tab)
            I = torch.concat((I_c, I_t),axis = 1)

            T_a = model.tab_a_block(img, tab)
            T_b = model.tab_b_block(img, tab)
            T = torch.concat((T_a, T_b),axis = 1)

            # if multiclass
            # Y = np.argmax(y.cpu().numpy(), axis=1)
            
            
            # Convert tensors to numpy and handle dimensions carefully
            tab_fts.append(tab.cpu().numpy())           # Keep tab_fts as is
            all_I.append(I.cpu().numpy())               # Append I features (variable column sizes)
            all_T.append(T.cpu().numpy())               # Append T features
            all_Y.append(y.cpu().numpy())               # Append labels

        # Convert lists to NumPy arrays and ensure consistent shape by concatenating along axis 0
        tab_fts_np = np.concatenate(tab_fts, axis=0)     # Stack rows for tab_fts
        all_I_np = np.concatenate(all_I, axis=0)         # Stack rows for all_I
        all_T_np = np.concatenate(all_T, axis=0)         # Stack rows for all_T
        all_Y_np = np.concatenate(all_Y, axis=0)         # Stack rows for all_Y

        # Ensure all_Y has a consistent shape, even if it has only one column
        if all_Y_np.ndim == 1:
            all_Y_np = all_Y_np[:, np.newaxis]           # Make sure all_Y is 2D (n, 1)

        # Create DataFrame for the first file (all_I, all_T, all_Y)
        df_1 = pd.DataFrame(
            data=np.hstack([all_I_np, all_T_np, all_Y_np]),  # Horizontally concatenate arrays
            columns=[f'I{i+1}' for i in range(all_I_np.shape[1])] +  # Name all_I columns
                    [f'T{i+1}' for i in range(all_T_np.shape[1])] +  # Name all_T columns
                    ['Y']  # Name the target column (all_Y)
        )
        
        # Create DataFrame for the second file (tab_fts, all_T)
        df_2 = pd.DataFrame(
            data=np.hstack([tab_fts_np, all_T_np]),  # Horizontally concatenate tab_fts and all_T
            columns=[f'tab_fts_col_{i+1}' for i in range(tab_fts_np.shape[1])] +  # Name tab_fts columns
                    [f'T{i+1}' for i in range(all_T_np.shape[1])]  # Name all_T columns
        )
print(nas)

In [ ]:
# load existing df
existing_df = pd.read_csv('/export/scratch2/ima/MultiFIX_GECCO25_code/scripts/gp_files/Multifeature/ft_part_part_single_False/train_tab_T.csv')

(df_2.sort_values('tab_fts_col_1').reset_index(drop = True)).equals(existing_df.sort_values('tab_fts_col_1').reset_index(drop = True).astype('float32'))

In [ ]:
#so T1 and T2 are switched
train = np.asarray(existing_df)
y_train = np.ascontiguousarray(train[:,-1])

np.ascontiguousarray(existing_df['T2']) == y_train

In [ ]:
# tab_a performance on A
data['y'] = get_y(data, 'tab_a')
train_loader, val_loader, test_loader = get_dataloaders(data, train_idx, test_idx, seed)
loss, auroc, bacc = eval_model(model.tab_a_block, test_loader, loss_fn)

print("*** Tab A block performance (frozen) on A***")
print("* Loss = ", loss)
print("* AUROC = ", auroc)
print("* BAcc = ", bacc)

# tab_b performance on B
data['y'] = get_y(data, 'tab_b')
train_loader, val_loader, test_loader = get_dataloaders(data, train_idx, test_idx, seed)
loss, auroc, bacc = eval_model(model.tab_b_block, test_loader, loss_fn)

print("*** Tab B block performance (frozen) on B ***")
print("* Loss = ", loss)
print("* AUROC = ", auroc)
print("* BAcc = ", bacc)

# tab_a performance on B
data['y'] = get_y(data, 'tab_b')
train_loader, val_loader, test_loader = get_dataloaders(data, train_idx, test_idx, seed)
loss, auroc, bacc = eval_model(model.tab_a_block, test_loader, loss_fn)

print("*** Tab A block performance (frozen) on B***")
print("* Loss = ", loss)
print("* AUROC = ", auroc)
print("* BAcc = ", bacc)

# tab_b performance on A
data['y'] = get_y(data, 'tab_a')
train_loader, val_loader, test_loader = get_dataloaders(data, train_idx, test_idx, seed)
loss, auroc, bacc = eval_model(model.tab_b_block, test_loader, loss_fn)

print("*** Tab B block performance (frozen) on A ***")
print("* Loss = ", loss)
print("* AUROC = ", auroc)
print("* BAcc = ", bacc)


In [ ]:
# checking if all weights are loaded
model = multifix_net_test(nas, OUT_SIZE)

training_args = TRAINING.split('_')

if len(training_args) == 3: # seeing that there is a part for loading weights, and freezing weights:
    train_load = training_args[1]
    train_freeze = training_args[2]
    print(train_load, train_freeze)

if train_load in ['comp', 'part']: # ft = fusion test
        # img c
    if model.img_c_block is not None:
        img_c_wts = torch.load(f'/export/scratch2/ima/MultiFIX_GECCO25_code/{MODEL_DIR[1:]}' + 'img_c' + str(1) + '.pth', map_location=DEVICE)
        model.img_c_block.load_state_dict(img_c_wts, strict=False)

    if model.img_t_block is not None:
        img_t_wts = torch.load(f'/export/scratch2/ima/MultiFIX_GECCO25_code/{MODEL_DIR[1:]}' + 'img_t' + str(1) + '.pth', map_location=DEVICE)
        model.img_t_block.load_state_dict(img_t_wts, strict=False)
    # tab a
    if model.tab_a_block is not None:
        tab_a_wts = torch.load(f'/export/scratch2/ima/MultiFIX_GECCO25_code/{MODEL_DIR[1:]}' + 'tab_a' + str(1) + '.pth', map_location=DEVICE)
        model.tab_a_block.load_state_dict(tab_a_wts)#, strict=False)

    if model.tab_b_block is not None:
        tab_b_wts = torch.load(f'/export/scratch2/ima/MultiFIX_GECCO25_code/{MODEL_DIR[1:]}' + 'tab_b' + str(1) + '.pth', map_location=DEVICE)
        model.tab_b_block.load_state_dict(tab_b_wts)#, strict=False)

results = model.img_c_block.load_state_dict(img_c_wts, strict=False)
print(f'missing keys: {results.missing_keys}')
print(f'unexpected keys: {results.unexpected_keys}')

results = model.img_t_block.load_state_dict(img_t_wts, strict=False)
print(f'missing keys: {results.missing_keys}')
print(f'unexpected keys: {results.unexpected_keys}')


results = model.tab_a_block.load_state_dict(tab_a_wts)#, strict=False)
print(f'missing keys: {results.missing_keys}')
print(f'unexpected keys: {results.unexpected_keys}')

results = model.tab_b_block.load_state_dict(tab_b_wts)#, strict=False)
print(f'missing keys: {results.missing_keys}')
print(f'unexpected keys: {results.unexpected_keys}')


freeze_list = [model.img_c_block, model.tab_a_block, model.img_t_block, model.tab_b_block]
for block in freeze_list:
    for param in block.parameters():
        param.requires_grad = False
print(f'Weights frozen: img_c + img_t + tab_a + tab_b')